# Identifying Fraudulent Activities — Modernized Workflow (2026)

This notebook rebuilds the original e-commerce fraud project with a leakage-safe, time-aware, and business-oriented machine-learning workflow.

**Business task:** At a user's first purchase, estimate the probability that the transaction is fraudulent so the company can approve it, send it to review, request stronger authentication, or block it.

## Main upgrades

- Replaces the H2O-only workflow with portable scikit-learn pipelines.
- Maps IP addresses to countries with a vectorized interval lookup instead of row-by-row searching.
- Replaces full-dataset device/IP counts with **historical counts available at event time**.
- Excludes raw identifiers and sensitive `sex` from the default model.
- Uses chronological splits rather than a random split.
- Compares a dummy baseline, balanced logistic regression, and histogram gradient boosting.
- Evaluates fraud ranking with **average precision** and ROC AUC, not accuracy alone.
- Calibrates probabilities on a dedicated period.
- Tunes the action threshold on a separate period using explicit false-positive and false-negative costs.
- Evaluates exactly once on an untouched future test period.
- Adds lift, calibration, drift, subgroup, error, and deployment analyses.

> A fraud score is a decision-support tool, not proof that a customer committed fraud. High-risk cases should follow a documented review and appeal process.

## 1. Environment

The notebook is designed for recent pandas and scikit-learn releases. It uses only widely available Python packages and does not require a Java/H2O service.

Uncomment the next line only when packages are missing.

In [ ]:
# %pip install -U pandas numpy scipy scikit-learn matplotlib seaborn joblib nbformat

In [ ]:
from __future__ import annotations

import json
import math
import os
import platform
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import scipy
import sklearn
from IPython.display import display

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.frozen import FrozenEstimator
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(context='notebook', style='whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

RANDOM_STATE = 42
TARGET = 'class'
USE_SENSITIVE_FEATURES = False  # `sex` is used for auditing, not training, by default.

print({
    'python': platform.python_version(),
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'scipy': scipy.__version__,
    'scikit_learn': sklearn.__version__,
})

## 2. Load the two source files

Expected files:

- `Fraud.csv`
- `IpAddress_to_Country.csv`

The loader checks environment variables first, then common local folders. This removes the hard-coded local path used by many older notebooks.

Optional environment variables:

```bash
export FRAUD_DATA_PATH=/path/to/Fraud.csv
export IP_COUNTRY_PATH=/path/to/IpAddress_to_Country.csv
```

In [ ]:
def find_data_file(filename: str, env_var: str) -> Path:
    candidates: list[Path] = []
    if os.getenv(env_var):
        candidates.append(Path(os.environ[env_var]).expanduser())

    cwd = Path.cwd()
    candidates.extend([
        cwd / filename,
        cwd / 'data' / filename,
        cwd.parent / 'data' / filename,
        Path('/mnt/data') / filename,
    ])

    for path in candidates:
        if path.exists() and path.is_file():
            return path.resolve()

    checked = '\n'.join(f'  - {p}' for p in candidates)
    raise FileNotFoundError(
        f'Could not locate {filename}. Checked:\n{checked}\n'
        f'Place the file beside this notebook, in data/, or set {env_var}.'
    )

fraud_path = find_data_file('Fraud.csv', 'FRAUD_DATA_PATH')
ip_country_path = find_data_file('IpAddress_to_Country.csv', 'IP_COUNTRY_PATH')

fraud_raw = pd.read_csv(fraud_path)
ip_country_raw = pd.read_csv(ip_country_path)

print(f'Fraud data: {fraud_path} — {fraud_raw.shape[0]:,} rows')
print(f'IP ranges:  {ip_country_path} — {ip_country_raw.shape[0]:,} rows')
display(fraud_raw.head())
display(ip_country_raw.head())

## 3. Validate the schema before analysis

Failing early is safer than silently training on renamed, missing, or incorrectly typed columns.

In [ ]:
FRAUD_REQUIRED = {
    'user_id', 'signup_time', 'purchase_time', 'purchase_value', 'device_id',
    'source', 'browser', 'sex', 'age', 'ip_address', TARGET,
}
IP_REQUIRED = {'lower_bound_ip_address', 'upper_bound_ip_address', 'country'}

missing_fraud = FRAUD_REQUIRED.difference(fraud_raw.columns)
missing_ip = IP_REQUIRED.difference(ip_country_raw.columns)
if missing_fraud or missing_ip:
    raise ValueError({
        'missing_fraud_columns': sorted(missing_fraud),
        'missing_ip_range_columns': sorted(missing_ip),
    })

schema_summary = pd.DataFrame({
    'dtype': fraud_raw.dtypes.astype(str),
    'missing_n': fraud_raw.isna().sum(),
    'missing_pct': fraud_raw.isna().mean(),
    'unique_n': fraud_raw.nunique(dropna=False),
}).sort_values('missing_pct', ascending=False)
display(schema_summary)

## 4. Clean records conservatively

Rules used here:

- Dates must parse successfully.
- The target must be 0 or 1.
- Purchase time cannot be before signup time.
- Purchase value cannot be negative.
- Age must be between 0 and 100.
- Exact duplicate rows are removed.

Unusual records are counted before removal so the data-quality impact is visible.

In [ ]:
def clean_fraud_data(raw: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = raw.copy()
    df['signup_time'] = pd.to_datetime(df['signup_time'], errors='coerce', utc=False)
    df['purchase_time'] = pd.to_datetime(df['purchase_time'], errors='coerce', utc=False)

    for col in ['purchase_value', 'age', 'ip_address', TARGET]:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    flags = pd.DataFrame(index=df.index)
    flags['bad_signup_time'] = df['signup_time'].isna()
    flags['bad_purchase_time'] = df['purchase_time'].isna()
    flags['bad_target'] = ~df[TARGET].isin([0, 1])
    flags['purchase_before_signup'] = df['purchase_time'] < df['signup_time']
    flags['negative_purchase_value'] = df['purchase_value'] < 0
    flags['invalid_age'] = ~df['age'].between(0, 100, inclusive='both')
    flags['exact_duplicate'] = df.duplicated(keep='first')

    report = pd.DataFrame({
        'issue': flags.columns,
        'rows': [int(flags[c].sum()) for c in flags.columns],
    })
    report['pct_of_raw'] = report['rows'] / max(len(df), 1)

    invalid = flags.any(axis=1)
    clean = df.loc[~invalid].copy()
    clean[TARGET] = clean[TARGET].astype('int8')
    clean['age'] = clean['age'].astype('float64')
    clean['purchase_value'] = clean['purchase_value'].astype('float64')
    clean['ip_address'] = clean['ip_address'].round().astype('Int64')

    # Standardize string categories while preserving unknown values.
    for col in ['device_id', 'source', 'browser', 'sex']:
        clean[col] = clean[col].astype('string').str.strip().fillna('Unknown')

    clean = clean.sort_values(['purchase_time', 'user_id']).reset_index(drop=True)
    return clean, report

fraud, quality_report = clean_fraud_data(fraud_raw)
display(quality_report)
print(f'Rows retained: {len(fraud):,} / {len(fraud_raw):,}')
print(f'Duplicate user IDs: {fraud["user_id"].duplicated().sum():,}')

## 5. Map IP addresses to countries efficiently

Each country row defines an inclusive numeric IP interval. For every transaction, `numpy.searchsorted` finds the last lower bound not exceeding the IP; the upper bound is then checked. Complexity is approximately `O(n log m)` rather than repeatedly scanning all ranges.

In [ ]:
def prepare_ip_ranges(raw: pd.DataFrame) -> pd.DataFrame:
    ranges = raw.copy()
    ranges['lower_bound_ip_address'] = pd.to_numeric(
        ranges['lower_bound_ip_address'], errors='coerce'
    )
    ranges['upper_bound_ip_address'] = pd.to_numeric(
        ranges['upper_bound_ip_address'], errors='coerce'
    )
    ranges['country'] = ranges['country'].astype('string').str.strip().fillna('Unknown')
    ranges = ranges.dropna(subset=['lower_bound_ip_address', 'upper_bound_ip_address'])
    ranges = ranges[
        ranges['lower_bound_ip_address'] <= ranges['upper_bound_ip_address']
    ].sort_values('lower_bound_ip_address').reset_index(drop=True)

    overlap = (
        ranges['lower_bound_ip_address'].iloc[1:].to_numpy()
        <= ranges['upper_bound_ip_address'].iloc[:-1].to_numpy()
    )
    if overlap.any():
        warnings.warn(f'{overlap.sum():,} adjacent IP ranges overlap; verify source data.')
    return ranges


def map_ip_to_country(ip: pd.Series, ranges: pd.DataFrame) -> pd.Series:
    lower = ranges['lower_bound_ip_address'].to_numpy(dtype='float64')
    upper = ranges['upper_bound_ip_address'].to_numpy(dtype='float64')
    countries = ranges['country'].astype(str).to_numpy()
    values = pd.to_numeric(ip, errors='coerce').to_numpy(dtype='float64')

    idx = np.searchsorted(lower, values, side='right') - 1
    valid_idx = idx >= 0
    safe_idx = np.clip(idx, 0, len(ranges) - 1)
    valid = valid_idx & np.isfinite(values) & (values <= upper[safe_idx])

    result = np.full(len(values), 'Unknown', dtype=object)
    result[valid] = countries[safe_idx[valid]]
    return pd.Series(result, index=ip.index, dtype='string')

ip_ranges = prepare_ip_ranges(ip_country_raw)
fraud['country'] = map_ip_to_country(fraud['ip_address'], ip_ranges)

print(f'Country match rate: {(fraud["country"] != "Unknown").mean():.2%}')
display(fraud[['ip_address', 'country']].head())

## 6. Build features available at decision time

A common leakage problem in the older approach is computing each device/IP's **final total count across the full dataset**. A transaction early in the year would then know about accounts that appear months later.

This version sorts transactions chronologically and uses `cumcount()`. Therefore:

- `device_prior_count` = earlier transactions using the same device.
- `ip_prior_count` = earlier transactions using the same IP.

For an online system, these values come from a feature store immediately before scoring. Raw `user_id`, `device_id`, and `ip_address` are never model inputs.

In [ ]:
def cyclic_encode(values: pd.Series, period: int, prefix: str) -> pd.DataFrame:
    angle = 2 * np.pi * values.astype(float) / period
    return pd.DataFrame({
        f'{prefix}_sin': np.sin(angle),
        f'{prefix}_cos': np.cos(angle),
    }, index=values.index)


def build_event_time_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.sort_values(['purchase_time', 'user_id']).copy()

    # Known at the first purchase.
    gap_seconds = (out['purchase_time'] - out['signup_time']).dt.total_seconds()
    out['time_since_signup_hours'] = gap_seconds / 3600
    out['log_time_since_signup_hours'] = np.log1p(out['time_since_signup_hours'].clip(lower=0))
    out['purchase_value_log'] = np.log1p(out['purchase_value'].clip(lower=0))

    out['purchase_is_weekend'] = out['purchase_time'].dt.dayofweek.ge(5).astype('int8')
    out['signup_is_weekend'] = out['signup_time'].dt.dayofweek.ge(5).astype('int8')
    out['instant_purchase_10s'] = gap_seconds.le(10).astype('int8')
    out['instant_purchase_1h'] = gap_seconds.le(3600).astype('int8')

    for frame in [
        cyclic_encode(out['purchase_time'].dt.hour, 24, 'purchase_hour'),
        cyclic_encode(out['purchase_time'].dt.dayofweek, 7, 'purchase_dow'),
        cyclic_encode(out['signup_time'].dt.hour, 24, 'signup_hour'),
        cyclic_encode(out['signup_time'].dt.dayofweek, 7, 'signup_dow'),
    ]:
        out = pd.concat([out, frame], axis=1)

    # Historical frequency features: strictly prior events only.
    device_key = out['device_id'].fillna('Unknown')
    ip_key = out['ip_address'].astype('string').fillna('Unknown')
    out['device_prior_count'] = out.groupby(device_key, sort=False).cumcount()
    out['ip_prior_count'] = out.groupby(ip_key, sort=False).cumcount()
    out['log_device_prior_count'] = np.log1p(out['device_prior_count'])
    out['log_ip_prior_count'] = np.log1p(out['ip_prior_count'])
    out['device_seen_before'] = out['device_prior_count'].gt(0).astype('int8')
    out['ip_seen_before'] = out['ip_prior_count'].gt(0).astype('int8')

    return out.reset_index(drop=True)

features_df = build_event_time_features(fraud)

display(features_df.head())
print('Engineered shape:', features_df.shape)

## 7. Exploratory analysis

Fraud data are usually imbalanced. Accuracy can therefore look excellent even when the model misses most fraud. We first inspect prevalence, time drift, and the support behind segment-level rates.

In [ ]:
prevalence = features_df[TARGET].mean()
print(f'Fraud prevalence: {prevalence:.2%} ({features_df[TARGET].sum():,} / {len(features_df):,})')

ax = features_df[TARGET].value_counts().sort_index().plot(kind='bar', figsize=(7, 4))
ax.set(title='Target counts', xlabel='Class (0 = legitimate, 1 = fraud)', ylabel='Transactions')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
monthly = (
    features_df.set_index('purchase_time')[TARGET]
    .resample('MS')
    .agg(['count', 'sum', 'mean'])
    .rename(columns={'sum': 'fraud_n', 'mean': 'fraud_rate'})
)
display(monthly)

ax = monthly['fraud_rate'].plot(marker='o', figsize=(11, 4))
ax.axhline(prevalence, linestyle='--', label='Overall prevalence')
ax.set(title='Fraud rate over purchase time', xlabel='Purchase month', ylabel='Fraud rate')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def wilson_interval(successes: pd.Series, totals: pd.Series, z: float = 1.96):
    totals = totals.astype(float)
    p = successes / totals
    denom = 1 + z**2 / totals
    center = (p + z**2 / (2 * totals)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * totals)) / totals) / denom
    return center - margin, center + margin


def segment_fraud_table(df: pd.DataFrame, column: str, min_n: int = 100) -> pd.DataFrame:
    table = (
        df.groupby(column, dropna=False)[TARGET]
        .agg(transactions='size', fraud_n='sum', fraud_rate='mean')
        .reset_index()
    )
    table = table[table['transactions'] >= min_n].copy()
    table['ci_low'], table['ci_high'] = wilson_interval(
        table['fraud_n'], table['transactions']
    )
    return table.sort_values(['fraud_rate', 'transactions'], ascending=[False, False])

for col in ['source', 'browser', 'country', 'sex']:
    print(f'\n{col.upper()}')
    display(segment_fraud_table(features_df, col, min_n=100).head(15))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.boxplot(data=features_df, x=TARGET, y='log_time_since_signup_hours', ax=axes[0], showfliers=False)
axes[0].set_title('Signup-to-purchase delay')

sns.boxplot(data=features_df, x=TARGET, y='log_device_prior_count', ax=axes[1], showfliers=False)
axes[1].set_title('Prior device activity')

sns.boxplot(data=features_df, x=TARGET, y='log_ip_prior_count', ax=axes[2], showfliers=False)
axes[2].set_title('Prior IP activity')

plt.tight_layout()
plt.show()

## 8. Modeling contract and leakage guardrails

### Prediction timestamp

The score is produced at the first purchase. Allowed features must exist by that moment.

### Default exclusions

- `class`: target.
- `user_id`: arbitrary identifier.
- `device_id`: raw high-cardinality identifier.
- `ip_address`: raw identifier; only mapped country and prior activity are used.
- `signup_time`, `purchase_time`: raw timestamps; only derived cyclic and elapsed-time features are used.
- `sex`: excluded from training by default and retained for subgroup auditing.

### Chronological partitions

1. **Train:** fit and compare algorithms.
2. **Calibration:** convert model scores into better probabilities.
3. **Threshold:** choose the business action threshold.
4. **Test:** untouched future period for final reporting.

This avoids training on future behavior and prevents threshold decisions from contaminating the test result.

In [ ]:
def chronological_split(
    df: pd.DataFrame,
    train_q: float = 0.60,
    calibration_q: float = 0.75,
    threshold_q: float = 0.85,
) -> dict[str, pd.DataFrame]:
    if not (0 < train_q < calibration_q < threshold_q < 1):
        raise ValueError('Quantiles must satisfy 0 < train < calibration < threshold < 1.')

    cutoffs = df['purchase_time'].quantile([train_q, calibration_q, threshold_q])
    t_train, t_cal, t_threshold = cutoffs.tolist()

    parts = {
        'train': df[df['purchase_time'] <= t_train].copy(),
        'calibration': df[(df['purchase_time'] > t_train) & (df['purchase_time'] <= t_cal)].copy(),
        'threshold': df[(df['purchase_time'] > t_cal) & (df['purchase_time'] <= t_threshold)].copy(),
        'test': df[df['purchase_time'] > t_threshold].copy(),
    }

    for name, part in parts.items():
        if part.empty or part[TARGET].nunique() < 2:
            raise ValueError(f'{name} split is empty or contains only one class.')
    return parts

parts = chronological_split(features_df)

split_summary = pd.DataFrame([
    {
        'split': name,
        'rows': len(part),
        'start': part['purchase_time'].min(),
        'end': part['purchase_time'].max(),
        'fraud_rate': part[TARGET].mean(),
        'fraud_n': int(part[TARGET].sum()),
    }
    for name, part in parts.items()
])
display(split_summary)

In [ ]:
NUMERIC_FEATURES = [
    'purchase_value', 'purchase_value_log', 'age',
    'time_since_signup_hours', 'log_time_since_signup_hours',
    'purchase_is_weekend', 'signup_is_weekend',
    'instant_purchase_10s', 'instant_purchase_1h',
    'purchase_hour_sin', 'purchase_hour_cos',
    'purchase_dow_sin', 'purchase_dow_cos',
    'signup_hour_sin', 'signup_hour_cos',
    'signup_dow_sin', 'signup_dow_cos',
    'device_prior_count', 'ip_prior_count',
    'log_device_prior_count', 'log_ip_prior_count',
    'device_seen_before', 'ip_seen_before',
]

CATEGORICAL_FEATURES = ['source', 'browser', 'country']
if USE_SENSITIVE_FEATURES:
    CATEGORICAL_FEATURES.append('sex')

MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

for forbidden in [TARGET, 'user_id', 'device_id', 'ip_address', 'signup_time', 'purchase_time']:
    assert forbidden not in MODEL_FEATURES

X = {name: part[MODEL_FEATURES].copy() for name, part in parts.items()}
y = {name: part[TARGET].copy() for name, part in parts.items()}

print(f'{len(MODEL_FEATURES)} model features')
print('Numeric:', NUMERIC_FEATURES)
print('Categorical:', CATEGORICAL_FEATURES)

## 9. Preprocessing and candidate models

Two preprocessing strategies are used:

- Logistic regression receives median-imputed, scaled numeric features and one-hot categorical features.
- Histogram gradient boosting receives median-imputed numeric values and bounded ordinal category codes. Scikit-learn's histogram implementation is efficient for large tabular datasets, supports missing values, and supports class weights.

Class weighting is applied **inside the estimator**, so the validation and test sets retain their natural fraud prevalence. No resampling touches the future evaluation periods.

In [ ]:
linear_preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), NUMERIC_FEATURES),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(
            handle_unknown='infrequent_if_exist',
            min_frequency=10,
            sparse_output=True,
        )),
    ]), CATEGORICAL_FEATURES),
])

ordinal_preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), NUMERIC_FEATURES),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1,
            encoded_missing_value=-1,
            max_categories=250,
        )),
    ]), CATEGORICAL_FEATURES),
])

categorical_mask = [False] * len(NUMERIC_FEATURES) + [True] * len(CATEGORICAL_FEATURES)

candidates = {
    'Dummy prevalence': Pipeline([
        ('preprocess', linear_preprocessor),
        ('model', DummyClassifier(strategy='prior')),
    ]),
    'Balanced logistic regression': Pipeline([
        ('preprocess', linear_preprocessor),
        ('model', LogisticRegression(
            class_weight='balanced',
            C=0.5,
            max_iter=2_000,
            solver='lbfgs',
            random_state=RANDOM_STATE,
        )),
    ]),
    'Balanced histogram gradient boosting': Pipeline([
        ('preprocess', ordinal_preprocessor),
        ('model', HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=250,
            max_leaf_nodes=31,
            min_samples_leaf=30,
            l2_regularization=1.0,
            categorical_features=categorical_mask,
            class_weight='balanced',
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=20,
            random_state=RANDOM_STATE,
        )),
    ]),
}

print('\n'.join(candidates))

## 10. Rolling model comparison on the training period

`TimeSeriesSplit` creates expanding training windows and later validation windows. The primary ranking metric is **average precision (AP)** because precision-recall behavior is informative when fraud is uncommon.

ROC AUC, log loss, and Brier loss are retained as complementary diagnostics. A model is selected only from the training period; calibration and threshold periods remain unseen.

In [ ]:
scoring = {
    'average_precision': 'average_precision',
    'roc_auc': 'roc_auc',
    'neg_log_loss': 'neg_log_loss',
    'neg_brier': 'neg_brier_score',
}

tscv = TimeSeriesSplit(n_splits=3)
cv_rows = []

for name, estimator in candidates.items():
    result = cross_validate(
        estimator,
        X['train'],
        y['train'],
        cv=tscv,
        scoring=scoring,
        n_jobs=1,
        error_score='raise',
    )
    row = {'model': name}
    for metric in scoring:
        values = result[f'test_{metric}']
        if metric.startswith('neg_'):
            values = -values
            metric_name = metric.removeprefix('neg_')
        else:
            metric_name = metric
        row[f'{metric_name}_mean'] = values.mean()
        row[f'{metric_name}_std'] = values.std(ddof=1)
    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).sort_values('average_precision_mean', ascending=False)
display(cv_results)

non_dummy = cv_results[~cv_results['model'].str.startswith('Dummy')]
SELECTED_MODEL_NAME = non_dummy.iloc[0]['model']
print('Selected model:', SELECTED_MODEL_NAME)

## 11. Fit and calibrate the selected model

The selected algorithm is fitted only on the training partition. A sigmoid calibrator then learns from the later calibration partition while the fitted base estimator is frozen.

Calibration changes the interpretation of the score, not its ordering. We compare raw and calibrated Brier/log losses before choosing a threshold.

In [ ]:
base_model = clone(candidates[SELECTED_MODEL_NAME])
base_model.fit(X['train'], y['train'])

raw_calibration_probability = base_model.predict_proba(X['calibration'])[:, 1]

calibrated_model = CalibratedClassifierCV(
    FrozenEstimator(base_model),
    method='sigmoid',
)
calibrated_model.fit(X['calibration'], y['calibration'])
calibrated_calibration_probability = calibrated_model.predict_proba(X['calibration'])[:, 1]

calibration_comparison = pd.DataFrame([
    {
        'version': 'raw',
        'brier_loss': brier_score_loss(y['calibration'], raw_calibration_probability),
        'log_loss': log_loss(y['calibration'], raw_calibration_probability, labels=[0, 1]),
        'average_precision': average_precision_score(y['calibration'], raw_calibration_probability),
        'roc_auc': roc_auc_score(y['calibration'], raw_calibration_probability),
    },
    {
        'version': 'calibrated',
        'brier_loss': brier_score_loss(y['calibration'], calibrated_calibration_probability),
        'log_loss': log_loss(y['calibration'], calibrated_calibration_probability, labels=[0, 1]),
        'average_precision': average_precision_score(y['calibration'], calibrated_calibration_probability),
        'roc_auc': roc_auc_score(y['calibration'], calibrated_calibration_probability),
    },
])
display(calibration_comparison)

fig, ax = plt.subplots(figsize=(7, 6))
CalibrationDisplay.from_predictions(
    y['calibration'], raw_calibration_probability,
    n_bins=10, strategy='quantile', name='Raw', ax=ax,
)
CalibrationDisplay.from_predictions(
    y['calibration'], calibrated_calibration_probability,
    n_bins=10, strategy='quantile', name='Calibrated', ax=ax,
)
ax.set_title('Calibration-period reliability')
plt.tight_layout()
plt.show()

## 12. Tune a business threshold on a separate period

A classification threshold is an operating policy, not an inherent property of the model.

The example cost assumptions below are deliberately editable:

- A false negative represents missed fraud and can include chargeback, merchandise, processing, and investigation losses.
- A false positive represents review friction, customer support, conversion loss, or unnecessary authentication.

Replace these values with finance/operations estimates before making a real decision.

In [ ]:
FALSE_NEGATIVE_COST = 500.0
FALSE_POSITIVE_COST = 5.0
REVIEW_CAPACITY_RATE = 0.03


def threshold_metrics(
    y_true: pd.Series | np.ndarray,
    probability: np.ndarray,
    thresholds: Iterable[float],
    false_negative_cost: float,
    false_positive_cost: float,
) -> pd.DataFrame:
    y_array = np.asarray(y_true, dtype=int)
    rows = []
    for threshold in thresholds:
        pred = (probability >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_array, pred, labels=[0, 1]).ravel()
        total_cost = fp * false_positive_cost + fn * false_negative_cost
        rows.append({
            'threshold': threshold,
            'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
            'precision': precision_score(y_array, pred, zero_division=0),
            'recall': recall_score(y_array, pred, zero_division=0),
            'f1': f1_score(y_array, pred, zero_division=0),
            'balanced_accuracy': balanced_accuracy_score(y_array, pred),
            'review_rate': pred.mean(),
            'total_cost': total_cost,
            'cost_per_1000': total_cost / len(y_array) * 1_000,
        })
    return pd.DataFrame(rows)

threshold_probability = calibrated_model.predict_proba(X['threshold'])[:, 1]
threshold_grid = np.unique(np.r_[np.linspace(0.001, 0.999, 500), 0.5])
threshold_table = threshold_metrics(
    y['threshold'], threshold_probability, threshold_grid,
    FALSE_NEGATIVE_COST, FALSE_POSITIVE_COST,
)

COST_THRESHOLD = float(
    threshold_table.loc[threshold_table['total_cost'].idxmin(), 'threshold']
)
CAPACITY_THRESHOLD = float(np.quantile(threshold_probability, 1 - REVIEW_CAPACITY_RATE))

policy_summary = threshold_metrics(
    y['threshold'],
    threshold_probability,
    [0.5, COST_THRESHOLD, CAPACITY_THRESHOLD],
    FALSE_NEGATIVE_COST,
    FALSE_POSITIVE_COST,
)
policy_summary.insert(0, 'policy', ['Default 0.5', 'Minimum expected cost', 'Review capacity'])
display(policy_summary)

print(f'Cost-optimal threshold: {COST_THRESHOLD:.4f}')
print(f'{REVIEW_CAPACITY_RATE:.1%} capacity threshold: {CAPACITY_THRESHOLD:.4f}')

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 5))
ax1.plot(threshold_table['threshold'], threshold_table['precision'], label='Precision')
ax1.plot(threshold_table['threshold'], threshold_table['recall'], label='Recall')
ax1.plot(threshold_table['threshold'], threshold_table['review_rate'], label='Review rate')
ax1.axvline(COST_THRESHOLD, linestyle='--', label='Cost threshold')
ax1.axvline(CAPACITY_THRESHOLD, linestyle=':', label='Capacity threshold')
ax1.set(xlabel='Threshold', ylabel='Rate', title='Threshold trade-offs on policy-tuning period')
ax1.legend(loc='upper right')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(threshold_table['threshold'], threshold_table['cost_per_1000'])
ax.axvline(COST_THRESHOLD, linestyle='--', label='Minimum expected cost')
ax.set(xlabel='Threshold', ylabel='Assumed cost per 1,000 transactions', title='Cost curve')
ax.legend()
plt.tight_layout()
plt.show()

## 13. Final untouched future-period evaluation

The test period has not been used for model selection, calibration, or threshold choice. Both threshold-independent and threshold-dependent metrics are reported.

In [ ]:
def evaluate_binary_model(
    y_true: pd.Series | np.ndarray,
    probability: np.ndarray,
    threshold: float,
    false_negative_cost: float = FALSE_NEGATIVE_COST,
    false_positive_cost: float = FALSE_POSITIVE_COST,
) -> dict[str, float]:
    y_array = np.asarray(y_true, dtype=int)
    pred = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_array, pred, labels=[0, 1]).ravel()
    return {
        'threshold': threshold,
        'prevalence': y_array.mean(),
        'roc_auc': roc_auc_score(y_array, probability),
        'average_precision': average_precision_score(y_array, probability),
        'brier_loss': brier_score_loss(y_array, probability),
        'log_loss': log_loss(y_array, probability, labels=[0, 1]),
        'precision': precision_score(y_array, pred, zero_division=0),
        'recall': recall_score(y_array, pred, zero_division=0),
        'f1': f1_score(y_array, pred, zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y_array, pred),
        'specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'review_rate': pred.mean(),
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
        'assumed_total_cost': fp * false_positive_cost + fn * false_negative_cost,
        'assumed_cost_per_1000': (
            (fp * false_positive_cost + fn * false_negative_cost) / len(y_array) * 1_000
        ),
    }

test_probability = calibrated_model.predict_proba(X['test'])[:, 1]
test_pred = (test_probability >= COST_THRESHOLD).astype(int)

test_metrics = pd.DataFrame([
    evaluate_binary_model(y['test'], test_probability, 0.5),
    evaluate_binary_model(y['test'], test_probability, COST_THRESHOLD),
    evaluate_binary_model(y['test'], test_probability, CAPACITY_THRESHOLD),
], index=['Default 0.5', 'Cost policy', 'Capacity policy'])

display(test_metrics.T)
print('\nClassification report — cost policy')
print(classification_report(y['test'], test_pred, digits=4, zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_predictions(
    y['test'], test_pred, labels=[0, 1], display_labels=['Legitimate', 'Fraud'],
    values_format=',d', ax=axes[0], colorbar=False,
)
axes[0].set_title(f'Confusion matrix — threshold {COST_THRESHOLD:.3f}')

precision, recall, _ = precision_recall_curve(y['test'], test_probability)
axes[1].plot(recall, precision, label=f'AP = {average_precision_score(y["test"], test_probability):.3f}')
axes[1].axhline(y['test'].mean(), linestyle='--', label='Prevalence baseline')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision–recall curve')
axes[1].legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(y['test'], test_probability, ax=ax, name=SELECTED_MODEL_NAME)
ax.plot([0, 1], [0, 1], linestyle='--')
ax.set_title('ROC curve — future test period')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
CalibrationDisplay.from_predictions(
    y['test'], test_probability, n_bins=10, strategy='quantile', ax=ax,
    name='Calibrated model',
)
ax.set_title('Probability calibration — future test period')
plt.tight_layout()
plt.show()

## 14. Lift and review-capacity analysis

Fraud teams often investigate only a small fraction of transactions. Lift asks whether the highest-risk transactions contain substantially more fraud than a random sample of the same size.

In [ ]:
def lift_table(y_true: pd.Series | np.ndarray, probability: np.ndarray, bins: int = 10) -> pd.DataFrame:
    frame = pd.DataFrame({'y': np.asarray(y_true), 'probability': probability})
    frame = frame.sort_values('probability', ascending=False).reset_index(drop=True)
    frame['risk_group'] = pd.qcut(
        frame.index + 1,
        q=bins,
        labels=[f'{i + 1}' for i in range(bins)],
    )
    grouped = frame.groupby('risk_group', observed=True).agg(
        transactions=('y', 'size'),
        fraud_n=('y', 'sum'),
        fraud_rate=('y', 'mean'),
        mean_score=('probability', 'mean'),
        min_score=('probability', 'min'),
    ).reset_index()
    grouped['population_pct'] = grouped['transactions'] / grouped['transactions'].sum()
    grouped['fraud_capture_pct'] = grouped['fraud_n'] / grouped['fraud_n'].sum()
    grouped['cumulative_population_pct'] = grouped['population_pct'].cumsum()
    grouped['cumulative_fraud_capture_pct'] = grouped['fraud_capture_pct'].cumsum()
    grouped['lift_vs_average'] = grouped['fraud_rate'] / frame['y'].mean()
    return grouped

lift = lift_table(y['test'], test_probability, bins=10)
display(lift)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(lift['risk_group'].astype(str), lift['fraud_rate'])
axes[0].axhline(y['test'].mean(), linestyle='--', label='Overall prevalence')
axes[0].set(title='Fraud rate by risk decile', xlabel='Risk decile (1 = highest)', ylabel='Fraud rate')
axes[0].legend()

axes[1].plot(lift['cumulative_population_pct'], lift['cumulative_fraud_capture_pct'], marker='o')
axes[1].plot([0, 1], [0, 1], linestyle='--', label='Random selection')
axes[1].set(title='Cumulative gains', xlabel='Cumulative transactions reviewed', ylabel='Cumulative fraud captured')
axes[1].legend()
plt.tight_layout()
plt.show()

## 15. Permutation importance on original business features

Permutation importance measures the reduction in test performance when one original column is shuffled. It is easier to interpret than split counts and works across both candidate model families.

Importance does **not** establish causality. Correlated features can share or mask importance.

In [ ]:
importance_result = permutation_importance(
    calibrated_model,
    X['test'],
    y['test'],
    scoring='average_precision',
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance = pd.DataFrame({
    'feature': MODEL_FEATURES,
    'importance_mean': importance_result.importances_mean,
    'importance_std': importance_result.importances_std,
}).sort_values('importance_mean', ascending=False)

display(importance.head(20))

plot_imp = importance.head(20).sort_values('importance_mean')
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(plot_imp['feature'], plot_imp['importance_mean'], xerr=plot_imp['importance_std'])
ax.set(title='Permutation importance — average precision decrease', xlabel='Importance')
plt.tight_layout()
plt.show()

## 16. Error analysis

Inspecting false negatives and false positives helps identify missing features, label problems, threshold issues, or process gaps. Raw identifiers are shown only for audit/debugging and must be access-controlled in a real system.

In [ ]:
audit_columns = [
    'user_id', 'purchase_time', 'purchase_value', 'country', 'source', 'browser', 'sex',
    'time_since_signup_hours', 'device_prior_count', 'ip_prior_count', TARGET,
]

audit = parts['test'][audit_columns].copy()
audit['fraud_probability'] = test_probability
audit['predicted_fraud'] = test_pred
audit['error_type'] = np.select(
    [
        (audit[TARGET] == 1) & (audit['predicted_fraud'] == 0),
        (audit[TARGET] == 0) & (audit['predicted_fraud'] == 1),
    ],
    ['false_negative', 'false_positive'],
    default='correct',
)

print('Highest-risk false positives')
display(
    audit[audit['error_type'] == 'false_positive']
    .sort_values('fraud_probability', ascending=False)
    .head(20)
)

print('Highest-score false negatives')
display(
    audit[audit['error_type'] == 'false_negative']
    .sort_values('fraud_probability', ascending=False)
    .head(20)
)

## 17. Subgroup audit

`sex` is not used by the default model, but it is retained to audit error rates. Country, source, and browser are also checked. Small groups are filtered because their rates are unstable.

A disparity is a signal for investigation—not automatic evidence of unlawful discrimination. Review label quality, sample size, customer exposure, operational policy, and local legal requirements.

In [ ]:
def subgroup_metrics(
    frame: pd.DataFrame,
    probability: np.ndarray,
    group: str,
    threshold: float,
    min_n: int = 100,
) -> pd.DataFrame:
    work = frame[[group, TARGET]].copy()
    work['probability'] = probability
    work['pred'] = (probability >= threshold).astype(int)

    rows = []
    for value, part in work.groupby(group, dropna=False):
        if len(part) < min_n or part[TARGET].nunique() < 2:
            continue
        tn, fp, fn, tp = confusion_matrix(part[TARGET], part['pred'], labels=[0, 1]).ravel()
        rows.append({
            group: value,
            'n': len(part),
            'prevalence': part[TARGET].mean(),
            'review_rate': part['pred'].mean(),
            'precision': precision_score(part[TARGET], part['pred'], zero_division=0),
            'recall': recall_score(part[TARGET], part['pred'], zero_division=0),
            'false_positive_rate': fp / (fp + tn) if fp + tn else np.nan,
            'average_precision': average_precision_score(part[TARGET], part['probability']),
        })
    return pd.DataFrame(rows).sort_values('n', ascending=False)

for group in ['sex', 'source', 'browser', 'country']:
    print(f'\n{group.upper()} AUDIT')
    display(subgroup_metrics(parts['test'], test_probability, group, COST_THRESHOLD, min_n=100).head(20))

## 18. Feature drift between training and future test periods

Population Stability Index (PSI) is included as a simple monitoring screen. It is a heuristic, not a statistical proof:

- below 0.10: usually stable;
- 0.10–0.25: watch;
- above 0.25: investigate.

Monitor score distributions, missingness, category changes, calibration, review rate, and delayed fraud outcomes in production.

In [ ]:
def _safe_distribution(values: pd.Series, categories: pd.Index, epsilon: float = 1e-6) -> np.ndarray:
    dist = values.value_counts(normalize=True, dropna=False).reindex(categories, fill_value=0).to_numpy()
    return np.clip(dist, epsilon, None)


def psi_categorical(reference: pd.Series, current: pd.Series) -> float:
    ref = reference.astype('string').fillna('Missing')
    cur = current.astype('string').fillna('Missing')
    categories = pd.Index(sorted(set(ref.unique()) | set(cur.unique())))
    p = _safe_distribution(ref, categories)
    q = _safe_distribution(cur, categories)
    return float(np.sum((q - p) * np.log(q / p)))


def psi_numeric(reference: pd.Series, current: pd.Series, bins: int = 10) -> float:
    ref = pd.to_numeric(reference, errors='coerce')
    cur = pd.to_numeric(current, errors='coerce')
    edges = np.unique(ref.quantile(np.linspace(0, 1, bins + 1)).to_numpy())
    if len(edges) < 3:
        return 0.0
    edges[0], edges[-1] = -np.inf, np.inf
    ref_bin = pd.cut(ref, edges, include_lowest=True).astype('string').fillna('Missing')
    cur_bin = pd.cut(cur, edges, include_lowest=True).astype('string').fillna('Missing')
    categories = pd.Index(sorted(set(ref_bin.unique()) | set(cur_bin.unique())))
    p = _safe_distribution(ref_bin, categories)
    q = _safe_distribution(cur_bin, categories)
    return float(np.sum((q - p) * np.log(q / p)))

psi_rows = []
for feature in MODEL_FEATURES:
    if feature in CATEGORICAL_FEATURES:
        value = psi_categorical(parts['train'][feature], parts['test'][feature])
    else:
        value = psi_numeric(parts['train'][feature], parts['test'][feature])
    psi_rows.append({'feature': feature, 'psi': value})

psi_table = pd.DataFrame(psi_rows).sort_values('psi', ascending=False)
psi_table['status'] = pd.cut(
    psi_table['psi'],
    bins=[-np.inf, 0.10, 0.25, np.inf],
    labels=['stable', 'watch', 'investigate'],
)
display(psi_table)

## 19. Save a reproducible model artifact and scored test sample

The saved object contains:

- fitted and calibrated preprocessing/model pipeline;
- selected threshold;
- feature contract;
- cost assumptions;
- split dates and package version metadata.

The production system must reproduce the event-time country and historical-count features before calling `predict_proba`.

In [ ]:
artifact = {
    'model': calibrated_model,
    'selected_model_name': SELECTED_MODEL_NAME,
    'decision_threshold': COST_THRESHOLD,
    'capacity_threshold': CAPACITY_THRESHOLD,
    'model_features': MODEL_FEATURES,
    'numeric_features': NUMERIC_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'uses_sensitive_features': USE_SENSITIVE_FEATURES,
    'cost_assumptions': {
        'false_negative_cost': FALSE_NEGATIVE_COST,
        'false_positive_cost': FALSE_POSITIVE_COST,
        'review_capacity_rate': REVIEW_CAPACITY_RATE,
    },
    'training_window': {
        'start': str(parts['train']['purchase_time'].min()),
        'end': str(parts['train']['purchase_time'].max()),
    },
    'test_window': {
        'start': str(parts['test']['purchase_time'].min()),
        'end': str(parts['test']['purchase_time'].max()),
    },
    'versions': {
        'python': platform.python_version(),
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
    },
}

MODEL_PATH = Path('fraud_detection_calibrated_model.joblib')
SCORED_TEST_PATH = Path('fraud_detection_scored_test.csv')

joblib.dump(artifact, MODEL_PATH)
audit.to_csv(SCORED_TEST_PATH, index=False)

print(f'Saved model artifact: {MODEL_PATH.resolve()}')
print(f'Saved scored test set: {SCORED_TEST_PATH.resolve()}')

In [ ]:
def score_engineered_transactions(engineered: pd.DataFrame, artifact_path: str | Path = MODEL_PATH) -> pd.DataFrame:
    saved = joblib.load(artifact_path)
    missing = set(saved['model_features']).difference(engineered.columns)
    if missing:
        raise ValueError(f'Missing engineered features: {sorted(missing)}')

    probability = saved['model'].predict_proba(engineered[saved['model_features']])[:, 1]
    result = pd.DataFrame(index=engineered.index)
    result['fraud_probability'] = probability
    result['send_to_review'] = probability >= saved['decision_threshold']
    result['capacity_policy_review'] = probability >= saved['capacity_threshold']
    return result

score_engineered_transactions(parts['test'].head())

## 20. Conclusions and recommended next steps

### What the model can support

- Rank first transactions by fraud risk.
- Route the riskiest transactions to manual review or step-up authentication.
- Quantify expected precision, recall, workload, lift, and cost under explicit assumptions.
- Track whether model probabilities remain calibrated over time.

### What the analysis does not prove

- A high score is not proof of illegal activity.
- Feature importance is not causal evidence.
- Historical labels may reflect past review policies and missed fraud.
- Offline test performance does not guarantee production impact.

### Production checklist

1. Confirm the exact scoring timestamp and availability of every feature.
2. Implement historical device/IP counters in an event-time feature store.
3. Version the IP-to-country database and handle unmatched IPs.
4. Validate false-positive and false-negative costs with finance, fraud operations, and support.
5. Select a policy based on both economics and review capacity.
6. Use reason codes and a human-review/appeal path for adverse actions.
7. Monitor input drift, score drift, calibration, review rate, subgroup metrics, and delayed outcomes.
8. Backtest on multiple future windows and shadow-test before enforcement.
9. Run a controlled experiment comparing the new policy with the existing process.
10. Retrain and recalibrate only through a documented, reproducible release process.

### Suggested extensions

- Add transaction network features from device–account–IP graphs.
- Add recent velocity windows such as prior events in 1 hour, 24 hours, and 7 days.
- Use delayed-label methods when chargeback outcomes arrive weeks later.
- Compare gradient boosting implementations such as LightGBM or CatBoost when organizational dependencies allow them.
- Add reject-inference or positive-unlabeled analysis if only reviewed transactions receive reliable labels.

## Design references

- Scikit-learn: precision–recall analysis for imbalanced classification  
  https://scikit-learn.org/stable/auto_examples/model_selection/plot_precision_recall.html
- Scikit-learn: probability calibration  
  https://scikit-learn.org/stable/modules/calibration.html
- Scikit-learn: decision-threshold tuning and cost-sensitive learning  
  https://scikit-learn.org/stable/auto_examples/model_selection/plot_cost_sensitive_learning.html
- Scikit-learn: `HistGradientBoostingClassifier`  
  https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html
- Scikit-learn: time-ordered cross-validation  
  https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html
- Imbalanced-learn: avoiding leakage when resampling  
  https://imbalanced-learn.org/stable/common_pitfalls.html